# 01 — Data: Tiny Shakespeare + character tokenizer

Step 1 of building a GPT from scratch. In this notebook we:

1. Download the dataset (~1 MB of Shakespeare plays)
2. Look at the text
3. Build a **character-level tokenizer** (each character → an integer)
4. Turn the whole text into a tensor and split it into train / validation
5. See what a training batch looks like — the exact input a GPT learns from

## 1. Download the dataset
Saved to `data/` (git-ignored, so the data never goes into the repo).

In [ ]:
from pathlib import Path
import urllib.request

DATA_DIR = Path("data")  # notebooks run from the project root (see .vscode/settings.json)
DATA_DIR.mkdir(exist_ok=True)
path = DATA_DIR / "tinyshakespeare.txt"

if not path.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, path)

text = path.read_text()
print(f"{len(text):,} characters")

## 2. Look at the text

In [ ]:
print(text[:500])

## 3. Character tokenizer

A model can only work with numbers. The simplest tokenizer: list every unique character
in the text, and give each one an id. Our *vocabulary* is just those characters.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print("vocab:", repr("".join(chars)))
print("vocab_size:", vocab_size)

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}  # string -> int
itos = {i: ch for i, ch in enumerate(chars)}  # int -> string


def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]


def decode(ids: list[int]) -> str:
    return "".join(itos[i] for i in ids)


print(encode("hello there"))
print(decode(encode("hello there")))

Real LLMs use *subword* tokenizers (e.g. GPT-2's BPE via `tiktoken`) — far fewer tokens
per sentence, but a ~50k vocabulary. Compare:

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
sample = "First Citizen: Before we proceed any further, hear me speak."
print("char tokens:", len(encode(sample)))
print("gpt2 tokens:", len(enc.encode(sample)), enc.encode(sample))

## 4. Encode everything and split train / validation
90% for training, the last 10% to check the model isn't just memorising.

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(data.shape, data.dtype)
print(f"train: {len(train_data):,}  val: {len(val_data):,}")

### Visualising the `data` tensor

`data` is just a long 1-D row of integers — one per character. Three views of it:

1. **The raw tensor** — the first tokens, each cell showing the character and the id it became
2. **Token frequencies** — how often each of the 65 ids appears
3. **The train / val split** — where the tensor is cut

In [ ]:
import matplotlib.pyplot as plt

# Shared chart styling: light surface, recessive axes, one blue for single-series charts
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, BLUE_LIGHT = "#2a78d6", "#eb6834", "#cde2fb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10,
})


def show_char(c: str) -> str:
    """Make whitespace visible."""
    return {"\n": "\\n", " ": "␣"}.get(c, c)

**1. The raw tensor.** Read left to right, top to bottom — this is literally what the model receives. Same character → same id, every time.

In [ ]:
rows, cols = 4, 16
fig, ax = plt.subplots(figsize=(12, 3.4))
for k in range(rows * cols):
    r, c = divmod(k, cols)
    tid = data[k].item()
    ax.add_patch(plt.Rectangle((c + 0.04, -r - 0.96), 0.92, 0.92, facecolor=BLUE_LIGHT, edgecolor=SURFACE, linewidth=2))
    ax.text(c + 0.5, -r - 0.38, show_char(itos[tid]), ha="center", va="center", fontsize=14, color=INK, family="monospace")
    ax.text(c + 0.5, -r - 0.78, tid, ha="center", va="center", fontsize=8, color=INK_2)
ax.set_xlim(0, cols)
ax.set_ylim(-rows, 0)
ax.axis("off")
ax.set_title(f"data[:{rows * cols}] — character on top, token id below", loc="left", color=INK)
plt.show()

**2. Token frequencies.** Space and lowercase vowels dominate; some ids (like `$`, `&`, `3`) almost never appear, so the model will see few examples of them.

In [ ]:
counts = torch.bincount(data, minlength=vocab_size)
order = torch.argsort(counts, descending=True)

fig, ax = plt.subplots(figsize=(12, 3.6))
ax.bar(range(vocab_size), counts[order].tolist(), width=0.8, color=BLUE)
ax.set_xticks(range(vocab_size), [show_char(itos[i.item()]) for i in order], fontsize=8, family="monospace")
ax.set_ylabel("occurrences")
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.yaxis.set_major_formatter(lambda v, _: f"{v / 1000:.0f}k")
top = counts[order[0]].item()  # label only the tallest bar
ax.annotate(f"space: {top:,}", (0, top), xytext=(10, -4), textcoords="offset points", va="top", fontsize=9, color=INK_2)
ax.set_title("How often each of the 65 tokens appears in data (sorted)", loc="left", color=INK)
plt.show()

**3. The train / val split.** One cut at 90%: the model trains on everything left of it and is evaluated on text it has never seen to the right.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 1.4))
ax.barh(0, len(train_data), color=BLUE, height=0.5, edgecolor=SURFACE, linewidth=2)
ax.barh(0, len(val_data), left=len(train_data), color=ORANGE, height=0.5, edgecolor=SURFACE, linewidth=2)
ax.text(len(train_data) / 2, 0, f"train  {len(train_data):,} tokens (90%)", ha="center", va="center", color="white")
ax.text(len(train_data) + len(val_data) / 2, 0, f"val {len(val_data):,}", ha="center", va="center", color="white", fontsize=9)
ax.set_xlim(0, len(data))
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.xaxis.set_major_formatter(lambda v, _: f"{v / 1e6:.1f}M")
ax.set_xlabel("position in data")
ax.set_title("data split into train_data and val_data", loc="left", color=INK)
plt.show()

## 5. What a training batch looks like

A GPT learns to predict the **next token**. We take random chunks of `block_size` tokens
(the context length). The target `y` is the same chunk shifted one position to the right,
so a single chunk holds `block_size` training examples.

In [ ]:
torch.manual_seed(1337)
block_size = 8  # context length
batch_size = 4  # chunks processed in parallel
device = "mps" if torch.backends.mps.is_available() else "cpu"


def get_batch(split: str):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


xb, yb = get_batch("train")
print("x:", xb.shape, "\n", xb)
print("y:", yb.shape, "\n", yb)

In [ ]:
# Every (context -> target) pair inside the first chunk
for t in range(block_size):
    context = xb[0, : t + 1].tolist()
    target = yb[0, t].item()
    print(f"{decode(context)!r:>12} -> {decode([target])!r}")

## Next

We now have `vocab_size`, `encode` / `decode`, and `get_batch`. Step 2 is the simplest
possible model — a **bigram** model that predicts the next character from only the current
one — followed by self-attention to make it a real transformer.